# Primary-9 budget confirmation (validation only)
Fresh A100 40GB sessions per model. Exactly two runs per family, no Traffic and no test. A 1000-step ceiling is a pilot bound, not a frozen final budget.


In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

assert (3, 11) <= sys.version_info[:2] <= (3, 12)
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
FAMILY = input("Model family (ttm or moirai1): ").strip()
assert FAMILY in ("ttm", "moirai1")
COMMIT = input("Full published primary-9 budget preparation commit SHA: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)

In [ ]:
ROOT = Path("/content") / ("tsfm-pilot-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/tsfm-budget-confirmation")
OUT = PERSIST / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
CACHE = PERSIST / "public-source-cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("Persistent results and checkpoints:", OUT)
print("Stale running.lock: confirm old process is dead before manually removing only that lock.")

In [ ]:
ENV = Path("/content") / ("venv-pilot-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", f"python{version}-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", f"requirements/{FAMILY}-gpu.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
probe = (
    "import sys,torch; print(sys.executable,sys.version,torch.__version__,torch.version.cuda);"
    "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(),"
    "torch.cuda.get_device_properties(0).total_memory)"
)
subprocess.run([str(PY), "-c", probe], check=True)
print("All model processes use:", PY, "; kernel imports no vendor package")

In [ ]:
PINNED = ROOT / "results/manifests/pilot/prepared_primary9.json"
prepared = ROOT / "data/prepared-budget-representatives.json"
if not prepared.exists():
    subprocess.run(
        [
            str(PY),
            "-m",
            "tsfm_crossover.data.pilot_data",
            "--names",
            "ETTh1",
            "Electricity",
            "--cache",
            str(CACHE),
            "--verify-against",
            str(PINNED),
            "--output",
            str(prepared),
        ],
        check=True,
    )
records = json.loads(prepared.read_text())
pins = json.loads(PINNED.read_text())
assert set(records) == {"ETTh1", "Electricity"}
for name, entry in records.items():
    assert entry["status"] == "ready_with_warnings"
    assert entry["qc"]["sha256"] == pins[name]["qc"]["sha256"]
print("Representative preparation validated; pilot rechecks actual file hashes.")

In [ ]:
BASE = [
    str(PY),
    "-m",
    "tsfm_crossover.experiments.pilot",
    "--config",
    "configs/pilot/budget_confirmation.yaml",
    "--prepared",
    str(PINNED),
    "--output",
    str(OUT),
    "--expected-commit",
    COMMIT,
]
subprocess.run(BASE, check=True)
plan = json.loads((OUT / "plan.json").read_text())
assert len(plan["conditions"]) == 4
jobs = [r for r in plan["conditions"] if r["family"] == FAMILY]
assert len(jobs) == 2
assert all(r["kind"] == "stability" and r["horizon"] == 96 for r in jobs)
assert all(r["dataset"] in {"ETTh1", "Electricity"} for r in jobs)
for row in jobs:
    print(row["dataset"], row["learning_rate"], row["id"])
print("At most 1000 steps each, FP32 batch 1, no LR sweep, no test.")

In [ ]:
assert input("Type RUN_BUDGET to execute these two validation-only conditions: ") == "RUN_BUDGET"
for row in jobs:
    subprocess.run(BASE + ["--condition-id", row["id"]], check=True)
    result = json.loads((OUT / row["id"] / "result.json").read_text())
    print(row["dataset"], result["status"], result["stopping_step"], result["best_validation_step"])
print("Completed conditions skip on rerun; checkpoints resume with matching identity.")

In [ ]:
from google.colab import files

archive = Path("/content") / f"{FAMILY}-budget-confirmation.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file() and p.suffix in {".json", ".csv"}:
            assert p.stat().st_size < 2_000_000, f"Unexpected large summary: {p.name}"
            z.write(p, p.relative_to(OUT))
files.download(str(archive))
print("Drive preserves checkpoints; export only JSON/CSV. Return ZIP for validation.")